# Bay Area County Housing Data Cleaning (2000–2025)

This notebook cleans the uploaded Zillow county ZHVI file for the group project.
Run cells from top to bottom (Jupyter: **Kernel → Restart Kernel and Run All Cells**).
Place the original CSV beside this notebook. Requires Python 3 and pandas.

**Scope:** nine Bay Area counties, January 2000–December 2025. ZHVI represents typical home value in nominal USD, not median sale price. The source is the all-homes, mid-tier, smoothed and seasonally adjusted series. Annual values below are our arithmetic summaries of monthly values, not a separate official annual Zillow series.

Source: https://www.zillow.com/research/data/
The uploaded source snapshot is preserved. No demographics are merged until Ben supplies compatible county/year data. Ben's subcounty geography is not assigned county housing values as if they were subcounty observations.


In [2]:
from pathlib import Path
import pandas as pd

INPUT = Path("County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv")
OUTPUT = Path("cleaned_outputs")
OUTPUT.mkdir(exist_ok=True)
START_YEAR, END_YEAR = 2000, 2025
COUNTIES = ["Alameda", "Contra Costa", "Marin", "Napa", "San Francisco",
            "San Mateo", "Santa Clara", "Solano", "Sonoma"]

raw = pd.read_csv(INPUT, dtype={"StateCodeFIPS": "string", "MunicipalCodeFIPS": "string"})
print("Original shape:", raw.shape)
print("Metadata columns:", raw.columns[:9].tolist())


Original shape: (3071, 330)
Metadata columns: ['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName', 'State', 'Metro', 'StateCodeFIPS', 'MunicipalCodeFIPS']


## 1. Select California's nine Bay Area counties

Filter on both state and county name. Preserve the original RegionID. Build a five-character county FIPS from the source's state and county codes; leading zeros matter. COUNTY omits the suffix “County”, matching Ben's county names. CTY_SUB is deliberately omitted because these are county-level observations.


In [5]:
required = {"RegionID", "RegionName", "RegionType", "State", "StateCodeFIPS", "MunicipalCodeFIPS"}
assert required.issubset(raw.columns), "Source metadata schema changed"
county = raw.loc[(raw["State"] == "CA") &
                 raw["RegionName"].isin([c + " County" for c in COUNTIES])].copy()
assert len(county) == 9 and county["RegionName"].nunique() == 9
assert county["RegionType"].eq("county").all()
county["COUNTY"] = county["RegionName"].str.replace(r" County$", "", regex=True)
county["COUNTY_FIPS"] = (county["StateCodeFIPS"].str.zfill(2) +
                          county["MunicipalCodeFIPS"].str.zfill(3))
assert county["COUNTY_FIPS"].str.fullmatch(r"06\d{3}").all()
assert county["COUNTY_FIPS"].nunique() == 9
print(county[["COUNTY", "COUNTY_FIPS", "RegionID"]].sort_values("COUNTY").to_string(index=False))


       COUNTY COUNTY_FIPS  RegionID
      Alameda       06001      1510
 Contra Costa       06013      3159
        Marin       06041       625
         Napa       06055       255
San Francisco       06075      3227
    San Mateo       06081      2842
  Santa Clara       06085      3136
       Solano       06095      1395
       Sonoma       06097      1396


## 2. Keep 2000–2025 and reshape monthly observations

Original: one row per county with hundreds of date columns. Cleaned: one row per county and month. Keep missing observations as blank/NaN; do not substitute zero or interpolate.


In [8]:
date_columns = [c for c in raw.columns if pd.notna(pd.to_datetime(c, format="%Y-%m-%d", errors="coerce"))]
selected_dates = [c for c in date_columns if START_YEAR <= int(c[:4]) <= END_YEAR]
expected_months = (END_YEAR - START_YEAR + 1) * 12
assert len(selected_dates) == expected_months

monthly = county.melt(id_vars=["RegionID", "COUNTY", "COUNTY_FIPS", "State"],
                      value_vars=selected_dates, var_name="DATE", value_name="ZHVI_USD")
monthly = monthly.rename(columns={"RegionID": "ZILLOW_REGION_ID", "State": "STATE"})
monthly["DATE"] = pd.to_datetime(monthly["DATE"], format="%Y-%m-%d")
monthly["YEAR"] = monthly["DATE"].dt.year
monthly["MONTH"] = monthly["DATE"].dt.month
monthly["ZHVI_USD"] = pd.to_numeric(monthly["ZHVI_USD"], errors="raise")
monthly["IS_MISSING"] = monthly["ZHVI_USD"].isna()
monthly["SOURCE"] = "Zillow ZHVI"
monthly = monthly[["YEAR", "MONTH", "DATE", "COUNTY", "COUNTY_FIPS", "STATE",
                   "ZHVI_USD", "IS_MISSING", "ZILLOW_REGION_ID", "SOURCE"]]
monthly = monthly.sort_values(["COUNTY_FIPS", "DATE"]).reset_index(drop=True)
assert len(monthly) == 9 * expected_months
assert not monthly.duplicated(["COUNTY_FIPS", "DATE"]).any()
assert monthly["ZHVI_USD"].dropna().gt(0).all()
print(monthly.head().to_string(index=False))


 YEAR  MONTH       DATE  COUNTY COUNTY_FIPS STATE      ZHVI_USD  IS_MISSING  ZILLOW_REGION_ID      SOURCE
 2000      1 2000-01-31 Alameda       06001    CA 265746.059169       False              1510 Zillow ZHVI
 2000      2 2000-02-29 Alameda       06001    CA 266693.517385       False              1510 Zillow ZHVI
 2000      3 2000-03-31 Alameda       06001    CA 268324.594908       False              1510 Zillow ZHVI
 2000      4 2000-04-30 Alameda       06001    CA 274461.552756       False              1510 Zillow ZHVI
 2000      5 2000-05-31 Alameda       06001    CA 283742.940728       False              1510 Zillow ZHVI


## 3. Inspect missing values

In the supplied snapshot, Contra Costa County is missing February 2007. Print the missing rows and their surrounding observations for review. A source gap is retained as a gap rather than silently repaired.


In [11]:
missing = monthly.loc[monthly["IS_MISSING"]].copy()
print("Missing observations:", len(missing))
print(missing[["COUNTY", "DATE", "ZHVI_USD"]].to_string(index=False))
for row in missing.itertuples():
    surrounding = monthly.loc[(monthly["COUNTY_FIPS"] == row.COUNTY_FIPS) &
                              monthly["DATE"].between(row.DATE - pd.Timedelta(days=65),
                                                       row.DATE + pd.Timedelta(days=65))]
    print(surrounding[["COUNTY", "DATE", "ZHVI_USD"]].to_string(index=False))


Missing observations: 1
      COUNTY       DATE  ZHVI_USD
Contra Costa 2007-02-28       NaN
      COUNTY       DATE      ZHVI_USD
Contra Costa 2006-12-31 659978.752302
Contra Costa 2007-01-31 669678.528279
Contra Costa 2007-02-28           NaN
Contra Costa 2007-03-31 640909.687016
Contra Costa 2007-04-30 620967.365983


## 4. Build the annual table

Calculate the arithmetic mean of available monthly ZHVI, keeping MONTHS_OBSERVED and IS_FULL_YEAR. For the missing-month year, the available-month mean uses 11 months and is explicitly flagged. ZHVI_FULL_YEAR_MEAN_USD is blank for that year, allowing strict 12-month comparisons. Do not label the 11-month value as a full-year mean. Values are nominal, with no inflation adjustment. Round exported dollar values to cents only after aggregation.


In [14]:
annual = monthly.groupby(["YEAR", "COUNTY", "COUNTY_FIPS", "STATE", "ZILLOW_REGION_ID"],
                         as_index=False).agg(
    ZHVI_AVAILABLE_MONTH_MEAN_USD=("ZHVI_USD", "mean"),
    MONTHS_OBSERVED=("ZHVI_USD", "count"))
annual["MONTHS_MISSING"] = 12 - annual["MONTHS_OBSERVED"]
annual["IS_FULL_YEAR"] = annual["MONTHS_OBSERVED"].eq(12)
annual["ZHVI_FULL_YEAR_MEAN_USD"] = annual["ZHVI_AVAILABLE_MONTH_MEAN_USD"].where(annual["IS_FULL_YEAR"])
annual["SOURCE"] = "Zillow ZHVI; annual mean calculated from monthly data"
annual = annual.sort_values(["COUNTY_FIPS", "YEAR"]).reset_index(drop=True)
assert len(annual) == 9 * (END_YEAR - START_YEAR + 1)
assert not annual.duplicated(["COUNTY_FIPS", "YEAR"]).any()
print("County-year rows:", len(annual))
print("Incomplete years:")
print(annual.loc[~annual["IS_FULL_YEAR"]].to_string(index=False))


County-year rows: 234
Incomplete years:
 YEAR       COUNTY COUNTY_FIPS STATE  ZILLOW_REGION_ID  ZHVI_AVAILABLE_MONTH_MEAN_USD  MONTHS_OBSERVED  MONTHS_MISSING  IS_FULL_YEAR  ZHVI_FULL_YEAR_MEAN_USD                                                SOURCE
 2007 Contra Costa       06013    CA              3159                  597879.171518               11               1         False                      NaN Zillow ZHVI; annual mean calculated from monthly data


## 5. Validate and export

Keys: COUNTY_FIPS + DATE (monthly), COUNTY_FIPS + YEAR (annual). Join to Ben on county FIPS + year where available; otherwise first validate county names and geographic level. Preserve FIPS as text when reading CSV in Excel or pandas.

The monthly CSV has 2,808 rows including the source gap. Annual CSV has 234 rows. The annual table is the main handoff for annual demographic comparison; it does not create missing demographic years.


In [17]:
coverage = monthly.groupby(["COUNTY", "COUNTY_FIPS"], as_index=False).agg(
    FIRST_DATE=("DATE", "min"), LAST_DATE=("DATE", "max"),
    MONTH_ROWS=("DATE", "size"), MONTHS_OBSERVED=("ZHVI_USD", "count"))
coverage["MONTHS_MISSING"] = coverage["MONTH_ROWS"] - coverage["MONTHS_OBSERVED"]
assert coverage["MONTH_ROWS"].eq(expected_months).all()
assert annual["MONTHS_OBSERVED"].sum() == monthly["ZHVI_USD"].notna().sum()

# Independently spot-check one annual result against its original wide source row.
check_county, check_year = "Alameda", 2020
original_values = county.loc[county["COUNTY"].eq(check_county),
                             [c for c in selected_dates if c.startswith(str(check_year))]].iloc[0]
result = annual.loc[annual["COUNTY"].eq(check_county) & annual["YEAR"].eq(check_year),
                    "ZHVI_AVAILABLE_MONTH_MEAN_USD"].iloc[0]
assert abs(result - original_values.mean()) < 1e-8

monthly.to_csv(OUTPUT / "bay_area_county_zhvi_monthly_2000_2025.csv", index=False,
               date_format="%Y-%m-%d", float_format="%.2f")
annual.to_csv(OUTPUT / "bay_area_county_zhvi_annual_2000_2025.csv", index=False, float_format="%.2f")
coverage.to_csv(OUTPUT / "coverage_report.csv", index=False, date_format="%Y-%m-%d")
missing.to_csv(OUTPUT / "missing_observations.csv", index=False, date_format="%Y-%m-%d")
saved = pd.read_csv(OUTPUT / "bay_area_county_zhvi_annual_2000_2025.csv", dtype={"COUNTY_FIPS":"string"})
assert len(saved) == len(annual) and saved["COUNTY_FIPS"].str.len().eq(5).all()
assert saved["ZHVI_FULL_YEAR_MEAN_USD"].isna().sum() == (~annual["IS_FULL_YEAR"]).sum()
print(coverage.to_string(index=False))
print("Validation passed. CSV files saved in:", OUTPUT.resolve())


       COUNTY COUNTY_FIPS FIRST_DATE  LAST_DATE  MONTH_ROWS  MONTHS_OBSERVED  MONTHS_MISSING
      Alameda       06001 2000-01-31 2025-12-31         312              312               0
 Contra Costa       06013 2000-01-31 2025-12-31         312              311               1
        Marin       06041 2000-01-31 2025-12-31         312              312               0
         Napa       06055 2000-01-31 2025-12-31         312              312               0
San Francisco       06075 2000-01-31 2025-12-31         312              312               0
    San Mateo       06081 2000-01-31 2025-12-31         312              312               0
  Santa Clara       06085 2000-01-31 2025-12-31         312              312               0
       Solano       06095 2000-01-31 2025-12-31         312              312               0
       Sonoma       06097 2000-01-31 2025-12-31         312              312               0
Validation passed. CSV files saved in: /Users/zhoujianhua/Desktop/Fall

In [19]:
# Clean county-level Zillow ZORI rental data
ZORI_INPUT = Path("County_zori_uc_sfrcondomfr_sm_month.csv")

zori_raw = pd.read_csv(
    ZORI_INPUT,
    dtype={"StateCodeFIPS": "string", "MunicipalCodeFIPS": "string"}
)

zori_county = zori_raw.loc[
    zori_raw["State"].eq("CA")
    & zori_raw["RegionName"].isin([c + " County" for c in COUNTIES])
].copy()

assert len(zori_county) == 9
assert zori_county["RegionName"].nunique() == 9
assert zori_county["RegionType"].eq("county").all()

zori_county["COUNTY"] = zori_county["RegionName"].str.replace(
    r" County$", "", regex=True
)
zori_county["COUNTY_FIPS"] = (
    zori_county["StateCodeFIPS"].str.zfill(2)
    + zori_county["MunicipalCodeFIPS"].str.zfill(3)
)

zori_dates = [
    c for c in zori_raw.columns
    if pd.notna(pd.to_datetime(c, format="%Y-%m-%d", errors="coerce"))
    and START_YEAR <= int(c[:4]) <= END_YEAR
]
assert zori_dates, "No monthly dates found within the project period"

zori_monthly = zori_county.melt(
    id_vars=["COUNTY", "COUNTY_FIPS"],
    value_vars=zori_dates,
    var_name="DATE",
    value_name="ZORI_USD_PER_MONTH"
)

zori_monthly["DATE"] = pd.to_datetime(zori_monthly["DATE"])
zori_monthly["YEAR"] = zori_monthly["DATE"].dt.year
zori_monthly["MONTH"] = zori_monthly["DATE"].dt.month
zori_monthly["ZORI_USD_PER_MONTH"] = pd.to_numeric(
    zori_monthly["ZORI_USD_PER_MONTH"], errors="raise"
)
zori_monthly["IS_MISSING"] = zori_monthly["ZORI_USD_PER_MONTH"].isna()
zori_monthly = zori_monthly.sort_values(
    ["COUNTY_FIPS", "DATE"]
).reset_index(drop=True)

assert not zori_monthly.duplicated(["COUNTY_FIPS", "DATE"]).any()
assert zori_monthly["ZORI_USD_PER_MONTH"].dropna().gt(0).all()

zori_annual = zori_monthly.groupby(
    ["YEAR", "COUNTY", "COUNTY_FIPS"], as_index=False
).agg(
    ZORI_AVAILABLE_MONTH_MEAN_USD=("ZORI_USD_PER_MONTH", "mean"),
    ZORI_MONTHS_OBSERVED=("ZORI_USD_PER_MONTH", "count")
)

zori_annual["ZORI_IS_FULL_YEAR"] = zori_annual["ZORI_MONTHS_OBSERVED"].eq(12)
zori_annual["ZORI_FULL_YEAR_MEAN_USD"] = (
    zori_annual["ZORI_AVAILABLE_MONTH_MEAN_USD"]
    .where(zori_annual["ZORI_IS_FULL_YEAR"])
)
zori_annual["SOURCE"] = "Zillow ZORI; annual mean of monthly rent levels"

zori_monthly.to_csv(
    OUTPUT / "bay_area_county_zori_monthly.csv",
    index=False, date_format="%Y-%m-%d", float_format="%.2f"
)
zori_annual.to_csv(
    OUTPUT / "bay_area_county_zori_annual.csv",
    index=False, float_format="%.2f"
)
zori_monthly.loc[zori_monthly["IS_MISSING"]].to_csv(
    OUTPUT / "zori_missing_observations.csv",
    index=False, date_format="%Y-%m-%d"
)

print("ZORI date range:", min(zori_dates), "to", max(zori_dates))
print("Monthly rows:", len(zori_monthly))
print("Missing monthly values:", zori_monthly["IS_MISSING"].sum())
print("\nIncomplete years:")
display(zori_annual.loc[~zori_annual["ZORI_IS_FULL_YEAR"]])

ZORI date range: 2015-01-31 to 2025-12-31
Monthly rows: 1188
Missing monthly values: 17

Incomplete years:


,YEAR,COUNTY,COUNTY_FIPS,ZORI_AVAILABLE_MONTH_MEAN_USD,ZORI_MONTHS_OBSERVED,ZORI_IS_FULL_YEAR,ZORI_FULL_YEAR_MEAN_USD,SOURCE
3,2015,Napa,06055,NaN,0,False,NaN,Zillow ZORI; annual mean of monthly rent levels
12,2016,Napa,06055,1987.031851,9,False,NaN,Zillow ZORI; annual mean of monthly rent levels
49,2020,San Francisco,06075,3304.307689,10,False,NaN,Zillow ZORI; annual mean of monthly rent levels


In [23]:
%pip install openpyxl


[notice] A new release of pip is available: 26.1.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [27]:
# Clean county-level annual FHFA HPI data
FHFA_INPUT = Path("hpi_at_county.xlsx")
# If you renamed the file, change the line above accordingly.

fhfa_preview = pd.read_excel(
    FHFA_INPUT, sheet_name="county", header=None, nrows=25
)

header_row = next(
    i for i, row in fhfa_preview.iterrows()
    if str(row.iloc[0]).strip() == "State"
    and "FIPS code" in row.tolist()
)

fhfa_raw = pd.read_excel(
    FHFA_INPUT,
    sheet_name="county",
    header=header_row,
    dtype={"FIPS code": "string"}
)

fhfa_raw = fhfa_raw.rename(columns={
    "State": "STATE",
    "County": "COUNTY",
    "FIPS code": "COUNTY_FIPS",
    "Year": "YEAR",
    "Annual Change (%)": "FHFA_ANNUAL_CHANGE_PCT",
    "HPI": "FHFA_HPI_NATIVE",
    "HPI with 1990 base": "FHFA_HPI_1990_BASE",
    "HPI with 2000 base": "FHFA_HPI_2000_BASE"
})

fhfa_raw["YEAR"] = pd.to_numeric(fhfa_raw["YEAR"], errors="coerce")

fhfa_annual = fhfa_raw.loc[
    fhfa_raw["STATE"].eq("CA")
    & fhfa_raw["COUNTY"].isin(COUNTIES)
    & fhfa_raw["YEAR"].between(START_YEAR, END_YEAR)
].copy()

fhfa_annual["YEAR"] = fhfa_annual["YEAR"].astype(int)
fhfa_annual["COUNTY_FIPS"] = fhfa_annual["COUNTY_FIPS"].str.zfill(5)

fhfa_metrics = [
    "FHFA_ANNUAL_CHANGE_PCT",
    "FHFA_HPI_NATIVE",
    "FHFA_HPI_1990_BASE",
    "FHFA_HPI_2000_BASE"
]

for column in fhfa_metrics:
    fhfa_annual[column] = pd.to_numeric(
        fhfa_annual[column].replace({".": None, "-": None}),
        errors="raise"
    )

fhfa_annual["SOURCE"] = (
    "FHFA annual county all-transactions developmental index; NSA"
)
fhfa_annual = fhfa_annual.sort_values(
    ["COUNTY_FIPS", "YEAR"]
).reset_index(drop=True)

assert fhfa_annual["COUNTY"].nunique() == 9
assert not fhfa_annual.duplicated(["COUNTY_FIPS", "YEAR"]).any()
assert fhfa_annual["COUNTY_FIPS"].str.fullmatch(r"06\d{3}").all()
assert fhfa_annual["FHFA_HPI_NATIVE"].dropna().gt(0).all()

fhfa_annual.to_csv(
    OUTPUT / "bay_area_county_fhfa_annual_2000_2025.csv",
    index=False, float_format="%.2f"
)

print("FHFA annual rows:", len(fhfa_annual))
print("Year range:", fhfa_annual["YEAR"].min(), "-", fhfa_annual["YEAR"].max())
print("\nMissing values:")
print(fhfa_annual[fhfa_metrics].isna().sum())
display(fhfa_annual.head())

FHFA annual rows: 234
Year range: 2000 - 2025

Missing values:
FHFA_ANNUAL_CHANGE_PCT    0
FHFA_HPI_NATIVE           0
FHFA_HPI_1990_BASE        0
FHFA_HPI_2000_BASE        0
dtype: int64


,STATE,COUNTY,COUNTY_FIPS,YEAR,FHFA_ANNUAL_CHANGE_PCT,FHFA_HPI_NATIVE,FHFA_HPI_1990_BASE,FHFA_HPI_2000_BASE,SOURCE
0,CA,Alameda,06001,2000,25.43,813.05,146.54,100.00,FHFA annual county all-transactions developmen...
1,CA,Alameda,06001,2001,14.82,933.58,168.26,114.82,FHFA annual county all-transactions developmen...
2,CA,Alameda,06001,2002,5.95,989.17,178.28,121.66,FHFA annual county all-transactions developmen...
3,CA,Alameda,06001,2003,5.72,1045.71,188.47,128.62,FHFA annual county all-transactions developmen...
4,CA,Alameda,06001,2004,12.34,1174.74,211.73,144.48,FHFA annual county all-transactions developmen...


In [29]:
# Combine annual ZHVI, ZORI and FHFA using county FIPS + year
join_keys = ["COUNTY_FIPS", "YEAR"]
county_reference = annual[["COUNTY_FIPS", "COUNTY"]].drop_duplicates()

for table in [annual, zori_annual, fhfa_annual]:
    assert not table.duplicated(join_keys).any()

    # Verify that county names and FIPS agree across datasets.
    pairs = table[["COUNTY_FIPS", "COUNTY"]].drop_duplicates()
    matched = pairs.merge(
        county_reference, on=["COUNTY_FIPS", "COUNTY"], how="inner"
    )
    assert len(matched) == len(pairs), "County identifiers do not match"

housing_annual = annual.rename(columns={
    "MONTHS_OBSERVED": "ZHVI_MONTHS_OBSERVED",
    "MONTHS_MISSING": "ZHVI_MONTHS_MISSING",
    "IS_FULL_YEAR": "ZHVI_IS_FULL_YEAR",
    "SOURCE": "ZHVI_SOURCE"
}).copy()

housing_annual = housing_annual.merge(
    zori_annual.drop(columns="COUNTY").rename(columns={"SOURCE": "ZORI_SOURCE"}),
    on=join_keys,
    how="left",
    validate="one_to_one"
)

housing_annual = housing_annual.merge(
    fhfa_annual.drop(columns=["COUNTY", "STATE"]).rename(
        columns={"SOURCE": "FHFA_SOURCE"}
    ),
    on=join_keys,
    how="left",
    validate="one_to_one"
)

housing_annual["HAS_ZORI"] = (
    housing_annual["ZORI_AVAILABLE_MONTH_MEAN_USD"].notna()
)
housing_annual["HAS_FHFA"] = housing_annual["FHFA_HPI_NATIVE"].notna()

assert len(housing_annual) == len(annual) == 234
assert not housing_annual.duplicated(join_keys).any()

housing_annual = housing_annual.sort_values(
    ["COUNTY_FIPS", "YEAR"]
).reset_index(drop=True)

housing_annual.to_csv(
    OUTPUT / "bay_area_county_housing_combined_2000_2025.csv",
    index=False, float_format="%.2f"
)

print("Combined annual rows:", len(housing_annual))
print("Years with available data by county:")
display(
    housing_annual.groupby("COUNTY")[
        ["ZHVI_FULL_YEAR_MEAN_USD", "ZORI_FULL_YEAR_MEAN_USD", "FHFA_HPI_NATIVE"]
    ].count().rename(columns={
        "ZHVI_FULL_YEAR_MEAN_USD": "ZHVI_COMPLETE_YEARS",
        "ZORI_FULL_YEAR_MEAN_USD": "ZORI_COMPLETE_YEARS",
        "FHFA_HPI_NATIVE": "FHFA_AVAILABLE_YEARS"
    })
)
print("Files saved to:", OUTPUT.resolve())
display(housing_annual.head())

Combined annual rows: 234
Years with available data by county:


,ZHVI_COMPLETE_YEARS,ZORI_COMPLETE_YEARS,FHFA_AVAILABLE_YEARS
COUNTY,,,
Alameda,26,11,26
Contra Costa,25,11,26
Marin,26,11,26
Napa,26,9,26
San Francisco,26,10,26
San Mateo,26,11,26
Santa Clara,26,11,26
Solano,26,11,26
Sonoma,26,11,26


Files saved to: /Users/zhoujianhua/Desktop/Fall2026/CSE291A-race:gender/project/data/County/bay_area_county_housing_notebook/cleaned_outputs


,YEAR,COUNTY,COUNTY_FIPS,STATE,ZILLOW_REGION_ID,ZHVI_AVAILABLE_MONTH_MEAN_USD,ZHVI_MONTHS_OBSERVED,ZHVI_MONTHS_MISSING,ZHVI_IS_FULL_YEAR,ZHVI_FULL_YEAR_MEAN_USD,...,ZORI_IS_FULL_YEAR,ZORI_FULL_YEAR_MEAN_USD,ZORI_SOURCE,FHFA_ANNUAL_CHANGE_PCT,FHFA_HPI_NATIVE,FHFA_HPI_1990_BASE,FHFA_HPI_2000_BASE,FHFA_SOURCE,HAS_ZORI,HAS_FHFA
0,2000,Alameda,06001,CA,1510,294741.319867,12,0,True,294741.319867,...,NaN,NaN,NaN,25.43,813.05,146.54,100.00,FHFA annual county all-transactions developmen...,False,True
1,2001,Alameda,06001,CA,1510,358952.636686,12,0,True,358952.636686,...,NaN,NaN,NaN,14.82,933.58,168.26,114.82,FHFA annual county all-transactions developmen...,False,True
2,2002,Alameda,06001,CA,1510,383939.187086,12,0,True,383939.187086,...,NaN,NaN,NaN,5.95,989.17,178.28,121.66,FHFA annual county all-transactions developmen...,False,True
3,2003,Alameda,06001,CA,1510,416418.596321,12,0,True,416418.596321,...,NaN,NaN,NaN,5.72,1045.71,188.47,128.62,FHFA annual county all-transactions developmen...,False,True
4,2004,Alameda,06001,CA,1510,472354.623268,12,0,True,472354.623268,...,NaN,NaN,NaN,12.34,1174.74,211.73,144.48,FHFA annual county all-transactions developmen...,False,True
